# PulmoSight — EfficientNet-B0 Pneumonia Detection Training & Evaluation

This self-contained notebook fine-tunes **EfficientNet-B0** on the Kaggle Chest X-Ray (Pneumonia) dataset and saves weights for PulmoSight.

**Colab Instructions**:
1. Select **GPU (T4)**: `Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4)`.
2. Run all cells sequentially. No external Python files required!

### Step 1: Install Dependencies & Setup Environment

In [ ]:
!pip install -q albumentations torchvision torch scikit-learn pyyaml matplotlib pillow

import os
import time
import numpy as np
from PIL import Image
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

### Step 2: Define Dataset Class

In [ ]:
class PneumoniaDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None, is_train=True):
        self.split_dir = Path(root_dir) / split
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label_name, label_idx in [("NORMAL", 0), ("PNEUMONIA", 1)]:
            class_dir = self.split_dir / label_name
            if class_dir.exists():
                for ext in ["*.jpeg", "*.jpg", "*.png", "*.JPEG", "*.JPG", "*.PNG"]:
                    for img_path in class_dir.glob(ext):
                        if not img_path.name.startswith("._"):
                            self.image_paths.append(str(img_path))
                            self.labels.append(label_idx)

        if self.transform is None:
            if is_train:
                self.transform = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.RandomRotation(degrees=10),
                    transforms.RandomHorizontalFlip(p=0.3),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ])
            else:
                self.transform = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        image = Image.open(img_path).convert("RGB")
        tensor_img = self.transform(image)
        return tensor_img, torch.tensor(label, dtype=torch.float32)

### Step 3: Build Model

In [ ]:
def build_model():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    for name, param in model.features.named_parameters():
        if not name.startswith("7") and not name.startswith("6"):
            param.requires_grad = False
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(in_features=1280, out_features=1),
    )
    return model

model = build_model().to(device)
print('EfficientNet-B0 initialized.')

### Step 4: Train Model with Weighted Loss & Early Stopping

In [ ]:
# Set path to dataset
DATA_DIR = 'data/chest_xray'  # Adjust path if uploaded elsewhere on Colab

full_train_dataset = PneumoniaDataset(DATA_DIR, split='train', is_train=True)
train_size = int(0.85 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_ds, val_ds = random_split(full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)

# Weighted Loss for class imbalance (1341 Normal / 3875 Pneumonia)
pos_weight = torch.tensor([0.346], device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0003, weight_decay=0.0001)

epochs = 10
best_val_loss = float('inf')
os.makedirs('weights', exist_ok=True)
checkpoint_path = 'weights/best_model.pth'

for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_ds)

    # Val
    model.eval()
    val_loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            outputs = model(images)
            val_loss_sum += criterion(outputs, labels).item() * images.size(0)
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = val_loss_sum / len(val_ds)
    val_acc = correct / total
    print(f'Epoch [{epoch}/{epochs}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'model_version': '1.0.0',
            'training_date': time.strftime('%Y-%m-%d'),
            'metrics': {'val_loss': val_loss, 'val_acc': val_acc}
        }, checkpoint_path)
        print(f'--> Saved new best checkpoint to {checkpoint_path}')

### Step 5: Evaluate on Test Set

In [ ]:
test_dataset = PneumoniaDataset(DATA_DIR, split='test', is_train=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

y_true, y_scores = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs).squeeze(-1).cpu().numpy()
        y_scores.extend(probs.tolist() if isinstance(probs, np.ndarray) and probs.ndim > 0 else [float(probs)])
        y_true.extend(labels.cpu().numpy().tolist())

y_true, y_scores = np.array(y_true), np.array(y_scores)
y_pred = (y_scores >= 0.5).astype(int)

print('--- TEST EVALUATION METRICS ---')
print(f'Accuracy: {accuracy_score(y_true, y_pred) * 100:.2f}%')
print(f'Recall (Pneumonia Sensitivity): {recall_score(y_true, y_pred, pos_label=1) * 100:.2f}%')
print(f'Precision (Pneumonia): {precision_score(y_true, y_pred, pos_label=1) * 100:.2f}%')
print(f'F1 Score: {f1_score(y_true, y_pred, pos_label=1):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_true, y_scores):.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_true, y_pred))
print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))